# jev-decision on Kaggle: BGE-M3, 2048 tokens, Bekko data

Trains the marker-scoring decision model (BGE-M3 + LoRA + decision head) on a large stream of the
**Bekko System One dataset** (`hotchpotch/bekko-system-one-dataset-v0`, 6.59M cases across 153 subsets).

**Before you start**
1. Notebook settings: **Accelerator = GPU T4 x2** (T4 is fine; P100 has no fast fp16), **Internet = On**.
2. Get the code in: either put the repo on GitHub and set `REPO_URL` below, or upload `jev-decision` as a Kaggle Dataset and attach it to this notebook.
3. Leave `SMOKE = True` for the first run (about 15 minutes). It checks that data loading, 2048-token memory use and training all work, and prints the real speed. Then set `SMOKE = False` and run the whole thing.

**What to expect.** A pool of 1M decisions is built, but a free Kaggle GPU cannot train on all of it: at 2048 tokens a T4 does very roughly 1 to 4 examples per second, so one 12-hour session sees about 50k to 150k examples. The trainer is therefore **time-budgeted**: it shuffles the pool, learning-rate decay follows the clock, and it stops at `TIME_BUDGET_H`. The speed printed in the smoke run tells you how much of the pool a full run will use. Only one GPU is used (the second T4 sits idle).

In [ ]:
SMOKE = True            # True: ~15 min end-to-end check. False: the real run.

# ---- real-run settings ------------------------------------------------------------
N_DECISIONS   = 600_000     # pool size. Smoke run: ~11 samples/s, so 9h trains ~350k; building 1M would cost ~1.5h and mostly go unused
TIME_BUDGET_H = 9.0         # training wall-clock; Kaggle sessions end at 12h, leave ~3h for data build + eval + packaging
MAX_LEN, HEAD_LEN = 2048, 512
BATCH, ACCUM = 4, 8         # effective batch 32. Lower BATCH if you hit CUDA out-of-memory.
LR_HEAD, LR_ENCODER = 3e-4, 2e-4   # untuned guesses; LoRA usually likes a higher LR than full fine-tuning
BACKBONE = "BAAI/bge-m3"

# ---- where the code comes from ----------------------------------------------------
REPO_URL = ""           # e.g. "https://github.com/YOUR_USER/jev-decision"; leave empty to use an attached Kaggle Dataset

if SMOKE:
    N_DECISIONS, TIME_BUDGET_H, VAL_MOD, CKPT_MIN, MAX_CONFIGS = 4000, 0.15, 20, 4, 12
else:
    VAL_MOD, CKPT_MIN, MAX_CONFIGS = 1000, 60, 0
print(dict(SMOKE=SMOKE, N_DECISIONS=N_DECISIONS, TIME_BUDGET_H=TIME_BUDGET_H, MAX_LEN=MAX_LEN))

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import urllib.request
try:
    urllib.request.urlopen("https://huggingface.co", timeout=10); print("huggingface.co reachable: OK")
except Exception as e:
    print("NO INTERNET -> turn Internet On in the notebook settings sidebar.", e)

## 1. Code

In [ ]:
import glob, os, shutil, subprocess
os.chdir("/kaggle/working")
if not os.path.exists("jev-decision"):
    for t in glob.glob("/kaggle/input/**/*.tar.gz", recursive=True):   # dataset stored as an un-extracted tarball
        os.makedirs("/kaggle/working/_unpacked", exist_ok=True)
        subprocess.run(["tar", "xzf", t, "-C", "/kaggle/working/_unpacked"], check=True); print("extracted", t)
    hits = [os.path.dirname(p) for p in glob.glob("/kaggle/**/pyproject.toml", recursive=True)
            if "name = \"jev-decision\"" in open(p).read()]
    if hits:
        shutil.copytree(hits[0], "jev-decision"); print("copied from attached dataset:", hits[0])
    elif REPO_URL:
        subprocess.run(["git", "clone", REPO_URL, "jev-decision"], check=True)
    else:
        raise SystemExit("No code found: attach the jev-decision Kaggle Dataset or set REPO_URL.")
os.chdir("/kaggle/working/jev-decision")
!pip install -q -e ".[data]"
# Kaggle ships torchao 0.10, which this peft refuses to import alongside; LoRA does not need torchao.
!pip uninstall -y -q torchao
import torch; print("torch", torch.__version__, "cuda:", torch.cuda.is_available())

## 2. Build the data
Streams subsets round-robin so all 153 are represented, writes straight to disk (flat memory), and keeps every state in val/test out of train.
The full 1M pool takes a while to stream; `train.jsonl` is written as it goes.

In [ ]:
args = f"--out data/large --n_train {N_DECISIONS} --val_mod {VAL_MOD} --max_configs {MAX_CONFIGS}"
!python scripts/build_bekko_large.py $args
!ls -la data/large && wc -l data/large/*.jsonl

In [ ]:
# Token-length profile at 2048 (val set): how often does the state still get cut?
!python scripts/validate_jsonl.py data/large/val.jsonl --tokenizer $BACKBONE --max_length $MAX_LEN --head_length $HEAD_LEN

## 3. Train
fp16 autocast with loss scaling (T4 has no bf16), LoRA rank 16, gradient checkpointing. The log prints `samples/s` and `progress`.
Every `CKPT_MIN` minutes it evaluates on val and saves `checkpoints/bge_m3_bekko/latest`, so a dropped session still leaves a usable checkpoint.
`--keep ends` keeps both the start and the end of long states.

In [ ]:
budget_min = int(TIME_BUDGET_H * 60)
!python -u -m jev.train --backbone $BACKBONE --train data/large/train.jsonl --val data/large/val.jsonl \
    --out checkpoints/bge_m3_bekko --max_length $MAX_LEN --head_length $HEAD_LEN --keep ends \
    --lora 16 --grad_ckpt --fp16 --batch_size $BATCH --grad_accum $ACCUM --epochs 1 \
    --lr_head $LR_HEAD --lr_encoder $LR_ENCODER --workers 2 \
    --time_limit_min $budget_min --checkpoint_every_min $CKPT_MIN 2>&1 | tee train.log

In [ ]:
# What did the run actually cover?
import re
log = open("train.log").read()
sp = re.findall(r"([\d.]+) samples/s", log)
if sp:
    rate = float(sp[-1]); n = sum(1 for _ in open("data/large/train.jsonl"))
    print(f"speed: {rate:.2f} samples/s")
    print(f"a {TIME_BUDGET_H:.1f}h run sees about {rate*3600*TIME_BUDGET_H:,.0f} examples = {rate*3600*TIME_BUDGET_H/n:.1%} of this pool")
for l in re.findall(r"\[eval @ step[^\n]*", log)[-3:]:
    print(l[:200])

## 4. Held-out evaluation
`test_typed` is the same LocalLLaMA/typed-decisions test split used in the Mac run, so the numbers are comparable (38.0% vs 31.9% chance). `test_bekko` is a broad sample from every subset's own test split.

In [ ]:
for name in ["test_typed", "test_bekko"]:
    print("=====", name)
    !python -m jev.evaluate --checkpoint checkpoints/bge_m3_bekko --data data/large/{name}.jsonl \
        --max_length $MAX_LEN --head_length $HEAD_LEN --keep ends --batch_size 4 --fp16 --summary --show 5

## 5. Package the checkpoint
Download `jev_checkpoint.tgz` from the notebook's **Output** tab (or it is saved with the notebook version). It holds the LoRA adapter, decision head and tokenizer, roughly 130 MB, and needs BGE-M3 at run time (you have it in corp).

In [ ]:
!cd checkpoints && tar czf /kaggle/working/jev_checkpoint.tgz bge_m3_bekko && ls -lh /kaggle/working/jev_checkpoint.tgz
!ls -R checkpoints/bge_m3_bekko | head -30

## Continue in corp
```
python -m jev.train --backbone /models/bge-m3 --init checkpoints/bge_m3_bekko \
   --train corp.jsonl --out checkpoints/corp --max_length 2048 --head_length 512 --keep tail \
   --lora 16 --grad_ckpt --bf16 --epochs 3 --lr_head 1e-4 --lr_encoder 5e-5
```